# Sample-side plots (`sample.ipynb`)

Reads `samples/<run>/` and `eval/<EVAL_NAME>/summary.json`; runs no model.

1. Pass/fail table (`passfail.md`)
2. **Per-event panels** (paired chains): fixed test events; columns truth | `sa_truth` | `sa_recon` | `sdedit` σ=0.5 | `sdedit` σ=2 | V1
3. **Unconditional gallery** (`full`): each generated event next to the nearest test event in feature space, **a visual reference only, not a pair**
4. Distributions: event-level scalar histograms (`full` vs test truth), profile mean ± 1σ, C2ST AUC per run
5. SDEdit curve: `sa_recon` (σ=0) → `sdedit` → `full` (σ=80)

Event metrics are cached in `<case>/viz/cache/`; figures are saved to `<case>/viz/figs_sample/`. Output of `selftest` is **synthetic data**.


In [ ]:
# parameters (run_notebook.py --set NAME=VALUE replaces these lines)
CONFIG = 'configs/v2.yaml'
CASE = 'v2a'
DEVICE = 'cpu'             # unused here; the notebook jobs always pass --set DEVICE
EVAL_NAME = 'main'         # eval/<EVAL_NAME>/summary.json; None = newest sample evaluation (not an ae-eval)
EVENTS = None              # None = viz.events of the config
PAIRED_RUNS = ['sa_truth', 'sa_recon', 'sdedit_0.5', 'sdedit_2']
FULL_RUN = 'full'
V1_DIR = None              # e.g. '<code dir>/artifacts/v1_samples/truth128' (written by jobs/v1_samples.sub)
N_GALLERY = 12
FIG_SUBDIR = 'figs_sample'


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'sparseshower').is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from sparseshower.common import config
from sparseshower.loader import case_dir, meta_for
from sparseshower import viz
c = config(CONFIG)
CASE_DIR = case_dir(c, CASE)
FIG_DIR = CASE_DIR / 'viz' / FIG_SUBDIR
FIG_DIR.mkdir(parents=True, exist_ok=True)
def save(fig, name):
    if fig is not None:
        fig.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')
        plt.show()
print('project  :', ROOT)
print('config   :', CONFIG, '| data', c['paths']['processed'])
print('case dir :', CASE_DIR, '(exists)' if CASE_DIR.exists() else '(MISSING)')
print('figures  ->', FIG_DIR)
if c.get('synthetic'):
    print('*** SYNTHETIC data (smoke config): nothing here is CORSIKA ***')

summary = viz.load_summary(c, CASE, EVAL_NAME)
if summary is None and EVAL_NAME is not None:
    print(f'eval/{EVAL_NAME} not found; using the newest evaluation instead')
    summary = viz.load_summary(c, CASE, None)
print('evaluation:', summary['name'] if summary else 'none yet', '| rungs:', list(summary['rungs']) if summary else [])
runs = viz.load_runs(c, CASE, PAIRED_RUNS + ([FULL_RUN] if FULL_RUN else []), V1_DIR)
for k, v in runs.items():
    print(f'run {k:<14} {len(v):>5} samples')


## 1. Pass/fail table


In [ ]:
if summary:
    print((CASE_DIR / 'eval' / summary['name'] / 'passfail.md').read_text())


## 2. Per-event panels (paired chains)

One figure per event: rows 1-2 are x-z / y-z photon sums (log10, shared colour scale, 5 decades below the truth maximum), row 3 a 3D scatter (sub-sampled), row 4 the longitudinal / radial profiles. The event's metrics table is printed below the figure.


In [ ]:
events = [int(e) for e in (EVENTS or c['viz']['events'] or meta_for(c)['split']['test'][:3])]
paired = {k: v for k, v in runs.items() if k != FULL_RUN}
for e in events:
    fig, table = viz.event_panel(c, e, paired)
    save(fig, f'2_event_{e}')
    viz.print_table(table)


## 3. Unconditional gallery (`full`)

Left: generated event. Right: the test event closest to it in standardized event-feature space.


In [ ]:
gen_metrics = None
if FULL_RUN in runs and runs[FULL_RUN]:
    gen_rows = runs[FULL_RUN]
    gen_metrics = viz.run_metrics(c, CASE, FULL_RUN, gen_rows)
    save(viz.gallery(c, CASE, FULL_RUN, gen_rows, gen_metrics, N_GALLERY), '3_gallery')
else:
    print('no unconditional run yet')


## 4. Distributions

Grey: all test truth; red: `full`. Titles give the W1/R0 from the evaluation (R0 = truth half B vs half A; 1 means truth-level agreement; threshold ≤ 2).


In [ ]:
test_ids, truth = viz.eval_truth(c, summary)
if gen_metrics is not None:
    save(viz.plot_distributions(c, gen_metrics, truth, summary, FULL_RUN), '4_distributions')
if summary:
    save(viz.plot_profiles(summary, truth), '4_profiles')
    save(viz.plot_c2st(summary), '4_c2st')


## 5. SDEdit curve

Left to right: `sa_recon` (σ=0, per event) → `sdedit` at each σ_s → `full` (σ=80, fully unconditional). Paired metrics are not defined for `full`.


In [ ]:
if summary:
    save(viz.plot_sdedit(summary), '5_sdedit')
